# Lesson 02 — Task Delegation vs Routing
### From Tool Selection to Agentic Planning

---

This tutorial builds directly on **Lesson 01 (Capability Discovery)**.
We now go deeper: instead of just routing, we teach a Planner Agent to reason, decide, and delegate.

By the end, you will be able to:
- Explain **why** routing and planning are fundamentally different
- Read and run both flows side-by-side
- Know **when** to use each pattern in real systems

---
## Part 1 — What You Already Built (Lesson 01 Recap)

In `01_capability_discovery`, the flow was:

```
User → Controller → [Discover] → [Route] → [Invoke] → Result
```

The controller:
- discovered agent cards from `/agent-card`
- asked Vertex AI: *"which agent name fits this input?"*
- called that agent's `/invoke`

This works well. But it has a ceiling. Let's understand where.

---
## Part 2 — The Problem with Pure Routing

Routing answers one question:

> "Which tool should I call?"

It is a **selection problem**.

Consider this user request:

```
"Calculate interest for 1000 and then add 10% tax"
```

A router sees this and asks: *which single agent handles this?*

There is no good answer — the task spans two domains.

The router either:
- gets confused and picks one,
- or needs complex conditional logic hardcoded by the developer.

**This is the ceiling of routing.**

---
## Part 3 — The Real Difference: One Table, One Truth

| Aspect | Discover → Route → Invoke | Discover → Plan → Delegate |
|---|---|---|
| Core question | Which tool? | How to solve + who does it? |
| Thinking | Selection | Reasoning + decision |
| Intelligence | Low | High |
| Controller role | Router / Switchboard | Planner Agent / Manager |
| Agent role | Tool | Independent worker / Employee |
| Multi-step tasks | Hard | Natural |
| Flexibility | Limited | High |
| Output of routing step | Agent name (string) | JSON plan with reason |

---

### One-line summary:

> **Routing = pick the best tool.**
>
> **Planning = decide how to solve the problem and assign work.**

---
## Part 4 — Mental Model

### Flow 1 — Controller as Switchboard
```
User calls in → switchboard connects to best extension → done.
```
- No memory of why
- No reasoning chain
- One shot decision

---

### Flow 2 — Controller as Manager
```
User describes a problem → manager understands it → assigns right employee → tracks reasoning.
```
- Reasoned decision
- Structured justification
- Easily extensible to multi-step

---
## Part 5 — Code Comparison: Routing vs Planning

Same agents. Same discovery. Different thinking layer.

### Lesson 01: Route step
```python
prompt = f"""
User: {user_input}
Agents: {agent_descriptions}
Return ONLY agent name.
"""
response = llm.invoke([HumanMessage(content=prompt)])
selected = response.content.strip()   # returns: "finance-agent"
```
👉 Output is a plain agent name. No reasoning. No structure.

---

### Lesson 02: Plan step
```python
prompt = f"""
You are a PLANNER AGENT.
Your job:
1. Understand the user request
2. Choose the best agent
3. Return JSON

User request: {user_input}
Available agents: {agent_descriptions}

Return ONLY JSON:
{{"agent": "<name>", "reason": "<why>"}}
"""
response = llm.invoke(prompt)
plan = json.loads(response.content)
# returns: {"agent": "finance-agent", "reason": "handles financial calculations"}
```
👉 Output is a structured plan. Reasoning is explicit and inspectable.

---
## Part 6 — Architecture Diagrams

### Lesson 01 Flow
```mermaid
flowchart TD
    U[User Input] --> D[Discover Node]
    D --> R[Route Node\nLLM picks agent name]
    R --> I[Invoke Node\nPOST /invoke]
    I --> O[Result]
```

---

### Lesson 02 Flow
```mermaid
flowchart TD
    U[User Input] --> D[Discover Node]
    D --> P[Planner Node\nLLM reasons + returns JSON plan]
    P --> G[Delegate Node\nassigns task per plan]
    G --> O[Result]
```

**Key visual difference:** `Route` is replaced by `Planner` (richer LLM step) and `Invoke` is replaced by `Delegate` (assignment, not just a call).

---
## Part 7 — Where the Gap Becomes Obvious

### Scenario: Multi-step request

```
"Calculate interest for 1000 and then add 10% tax"
```

#### ❌ Route Flow fails
```python
# Router picks ONE agent — ignores second task
"finance-agent"   # tax step silently dropped
```

#### ✅ Plan Flow handles it
```json
{
  "steps": [
    {"agent": "finance-agent", "task": "calculate interest for 1000"},
    {"agent": "math-agent",    "task": "add 10% tax to result"}
  ]
}
```
Multi-agent workflow becomes natural when the planner returns structured steps.

---
## Part 8 — When to Use Each Pattern

### Use Route → Invoke when:
- Single-step tasks
- Tool-like agent usage
- Speed matters more than reasoning
- Low-stakes classification

### Use Plan → Delegate when:
- Multi-step workflows possible
- Multiple agents with overlapping domains
- You need auditable decisions (reason field)
- Enterprise or production-grade systems
- Agents are treated as independent workers, not tools

---
## Part 9 — Full Implementation (Lesson 02)

Below is the complete runnable Lesson 02 code.

**Before running:** ensure `finance_agent.py` (port `8001`) and `math_agent.py` (port `8002`) from `01_capability_discovery` are still running.

In [ ]:
import requests
import json
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph
from langchain_google_vertexai import ChatVertexAI

In [ ]:
# Vertex AI LLM — gemini-1.5-pro for stronger planning reasoning
llm = ChatVertexAI(
    model="gemini-1.5-pro",
    temperature=0
)

In [ ]:
# Typed State — more structured than Lesson 01's plain dict
class State(TypedDict, total=False):
    user_input: str
    agents: List[Dict[str, Any]]
    plan: Dict[str, Any]
    result: Dict[str, Any]

AGENT_URLS = [
    "http://localhost:8001",
    "http://localhost:8002"
]

In [ ]:
# Discovery — same pattern as Lesson 01
def discover_agents():
    agents = []
    for url in AGENT_URLS:
        try:
            res = requests.get(f"{url}/agent-card").json()
            agents.append(res)
        except Exception as e:
            print(f"[ERROR] {url}: {e}")
    return agents

In [ ]:
# Planner — KEY difference from Lesson 01
# LLM now reasons, not just classifies
def plan_and_delegate(user_input, agents):
    agent_descriptions = "\n".join([
        f"{a['name']}: {a['description']} | skills: {[s['name'] for s in a['skills']]}"
        for a in agents
    ])

    prompt = f"""
You are a PLANNER AGENT.

Your job:
1. Understand the user request
2. Choose the best agent
3. Return JSON

User request:
{user_input}

Available agents:
{agent_descriptions}

Return ONLY JSON:
{{
  "agent": "<agent_name>",
  "reason": "<why>"
}}
"""

    response = llm.invoke(prompt)
    return response.content

In [ ]:
# Graph Nodes
def discovery_node(state: State):
    print("\n--- DISCOVERY ---")
    state["agents"] = discover_agents()
    print(f"Found {len(state['agents'])} agents: {[a['name'] for a in state['agents']]}")
    return state


def planner_node(state: State):
    print("\n--- PLANNER ---")
    raw_plan = plan_and_delegate(state["user_input"], state["agents"])
    print("RAW PLAN FROM LLM:", raw_plan)

    # Strip markdown code fences if LLM wraps in ```json
    clean = raw_plan.strip().strip("```json").strip("```").strip()
    try:
        state["plan"] = json.loads(clean)
    except Exception:
        state["plan"] = {"agent": "finance-agent", "reason": "fallback"}

    print("PARSED PLAN:", state["plan"])
    return state


def delegation_node(state: State):
    print("\n--- DELEGATION ---")
    selected = state["plan"]["agent"]
    reason   = state["plan"].get("reason", "N/A")
    print(f"Delegating to: {selected}\nReason: {reason}")

    agent = next(
        (a for a in state["agents"] if a["name"] == selected), None
    )

    if agent is None:
        state["result"] = {"error": f"Agent '{selected}' not found in discovered agents"}
        return state

    res = requests.post(
        agent["endpoint"],
        json={"input": state["user_input"]}
    )
    state["result"] = res.json()
    print("RESULT:", state["result"])
    return state

In [ ]:
# Build and compile LangGraph
graph = StateGraph(State)

graph.add_node("discover", discovery_node)
graph.add_node("planner",  planner_node)
graph.add_node("delegate", delegation_node)

graph.set_entry_point("discover")
graph.add_edge("discover", "planner")
graph.add_edge("planner",  "delegate")

app = graph.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

---
## Part 10 — Run: Finance Task (delegated to finance-agent)

In [ ]:
result = app.invoke({"user_input": "calculate interest for 1000"})
print("\nFINAL RESULT:", result["result"])
print("PLAN REASON: ", result["plan"]["reason"])

---
## Part 11 — Run: Math Task (delegated to math-agent)

In [ ]:
result2 = app.invoke({"user_input": "multiply 4 and 5"})
print("\nFINAL RESULT:", result2["result"])
print("PLAN REASON: ", result2["plan"]["reason"])

---
## Key Takeaways

| | Lesson 01 | Lesson 02 |
|---|---|---|
| LLM output | `"finance-agent"` (string) | `{"agent": "...", "reason": "..."}` (JSON) |
| State | plain dict | `TypedDict` (typed, inspectable) |
| Node names | `discover → route → invoke` | `discover → planner → delegate` |
| Pattern | Tool selection | Agent delegation |
| Multi-step capable | No | Yes (extend plan schema) |

---

**Next natural step:** extend `plan` to return a `steps` list and loop the delegation node across all steps — that is Lesson 03 territory.